### 1. Hardware info: GPU, disk, CPU, RAM, Python

In [ ]:
import os, subprocess
os.makedirs("/kaggle/working", exist_ok=True)
os.makedirs("/kaggle/tmp", exist_ok=True)
HW = "/kaggle/working/hardware.txt"
cmds = [
    "nvidia-smi",
    "df -h /kaggle/tmp /kaggle/working",
    "lscpu | head -20",
    "free -g",
    "python --version",
]
with open(HW, "a") as f:
    for c in cmds:
        out = subprocess.run(c, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True).stdout
        block = f"$ {c}\n{out}\n"
        print(block)
        f.write(block)


### 2. Clone (or pull) the repo

In [ ]:
!if [ -d /kaggle/working/repo/.git ]; then git -C /kaggle/working/repo pull; else git clone https://github.com/irfy2k/alethia-video-indexing /kaggle/working/repo; fi
!git -C /kaggle/working/repo log -1 --oneline


### 3. Install requirements, check torch and CUDA

In [ ]:
!pip install -q -r /kaggle/working/repo/requirements.txt
import torch
print("torch version:", torch.__version__)
print("torch.cuda.is_available():", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("WARNING: CUDA not available. Enable the GPU accelerator (T4) in notebook settings.")


### 4. Prep: run prepare.py (or fetch yolo11l.pt directly)

In [ ]:
import os, subprocess
REPO = "/kaggle/working/repo"
if os.path.exists(os.path.join(REPO, "prepare.py")):
    cmd = ["python", "prepare.py"]
else:
    cmd = ["python", "-c", "from ultralytics import YOLO; YOLO('yolo11l.pt')"]
print("running:", " ".join(cmd), "in", REPO)
r = subprocess.run(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(r.stdout)
print("returncode:", r.returncode)
if r.returncode != 0:
    print("WARNING: prep step failed; check output above before continuing.")


### 5. Load HF token from Kaggle Secrets (value is never printed)

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
print("HF_TOKEN loaded:", bool(os.environ.get("HF_TOKEN")))


### 6. Download 50 FineVideo parquet shards and verify

In [ ]:
import os, glob, shutil, time
from huggingface_hub import snapshot_download
DATA = "/kaggle/tmp/finevideo"
os.makedirs(DATA, exist_ok=True)
free_gb = shutil.disk_usage("/kaggle/tmp").free / 1e9
print(f"free disk on /kaggle/tmp: {free_gb:.1f} GB")
if free_gb < 30:
    print(f"WARNING: only {free_gb:.1f} GB free on /kaggle/tmp (under 30 GB). The download may run out of space.")
t0 = time.time()
snapshot_download(
    repo_id="HuggingFaceFV/finevideo",
    repo_type="dataset",
    allow_patterns=["data/train-000[0-4]*"],
    local_dir=DATA,
    max_workers=16,
)
print(f"download took {time.time() - t0:.0f} s")
files = sorted(glob.glob(os.path.join(DATA, "data", "*.parquet")))
assert len(files) == 50, f"expected 50 parquet files, found {len(files)}"
total_gb = sum(os.path.getsize(f) for f in files) / 1e9
print(f"{len(files)} parquet files, {total_gb:.2f} GB total")


### 7. Timed full run of run.py (output streamed to cell, run.log and hardware.txt)

In [ ]:
import os, subprocess, time
REPO = "/kaggle/working/repo"
OUT = "/kaggle/working/results.jsonl"
LOG = "/kaggle/working/run.log"
HW = "/kaggle/working/hardware.txt"
cmd = ["python", "run.py", "--input", "/kaggle/tmp/finevideo/data", "--output", OUT]
env = dict(os.environ, PYTHONUNBUFFERED="1")
start = time.time()
with open(LOG, "w") as log:
    proc = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, env=env)
    for line in proc.stdout:
        print(line, end="", flush=True)
        log.write(line)
    rc = proc.wait()
end = time.time()
elapsed = end - start
h, rem = divmod(int(elapsed), 3600)
m, s = divmod(rem, 60)
hms = f"{h}:{m:02d}:{s:02d}"
print(f"\nrun.py returncode={rc}")
print(f"elapsed={elapsed:.1f} s ({hms})")
with open(HW, "a") as f:
    f.write(f"full run: start={start:.0f} end={end:.0f} elapsed={elapsed:.1f} s ({hms}) returncode={rc}\n")
if rc != 0:
    print("WARNING: run.py exited non-zero. See /kaggle/working/run.log.")


### 8. Sanity checks on results, gzip the output

In [ ]:
import gzip, json, os, shutil
OUT = "/kaggle/working/results.jsonl"
GZ = OUT + ".gz"
if not os.path.exists(OUT):
    raise FileNotFoundError(f"{OUT} not found; run.py did not write results")
n_lines = n_err = n_bad = 0
videos = set()
key = None
with open(OUT) as f:
    for line in f:
        if not line.strip():
            continue
        n_lines += 1
        try:
            obj = json.loads(line)
        except json.JSONDecodeError:
            n_bad += 1
            continue
        if not isinstance(obj, dict):
            continue
        if obj.get("error") or obj.get("status") == "error":
            n_err += 1
        if key is None:
            key = next((k for k in ("video_id", "video", "video_path", "file", "filename") if k in obj), None)
        if key is not None:
            videos.add(str(obj.get(key)))
print(f"result lines:        {n_lines}")
print(f"unparseable lines:   {n_bad}")
print(f"error lines:         {n_err}")
print(f"distinct videos:     {len(videos)} (key: {key or 'not found'})")
with open(OUT, "rb") as src, gzip.open(GZ, "wb", compresslevel=9) as dst:
    shutil.copyfileobj(src, dst)
for p in (OUT, GZ, "/kaggle/working/run.log", "/kaggle/working/hardware.txt"):
    if os.path.exists(p):
        print(f"{p}: {os.path.getsize(p) / 1e6:.2f} MB")


Download `results.jsonl.gz`, `run.log` and `hardware.txt` from the Output panel (they are in `/kaggle/working`).